## V4 KOLF Graph：扩展 control 共表达图与 OOD 安全 DEG 节点

本 notebook 接在 **V4_KOLF_Data.ipynb** 后运行，保留 V1 的 SEACells、共表达建图和 cells.pkl 结构。

图节点集合为：

```text
control HVG5000
+ 当前 KOLF 表达可用的 VCC 靶基因
+ KOLF 训练靶基因
+ 训练靶点的显著 top DEG（仅 VCC 输出面板内）
```

DEG 选择明确排除 OOD 测试靶点。图和输出均使用 V4 固定文件名，不覆盖 V1/V2/V3。

In [ ]:
import scanpy as sc 
import os
import json
import random
import datetime
import pandas as pd
import SEACells
import numpy as np
import matplotlib
from statistics import mode
import matplotlib.pyplot as plt
import seaborn as sns
import sys 
from scipy.sparse import issparse

PROJECT_DIR = r"E:\VSCODE\PrePR-CT\Cell-Type-Specific-Graphs-main"
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
import utils
from importlib import reload
reload(utils)
from utils import *

%matplotlib inline
sns.set_style('ticks')
matplotlib.rcParams['figure.figsize'] = [4, 4]
matplotlib.rcParams['figure.dpi'] = 100

# =============================================================================
# V4 configuration
# =============================================================================
data_path = os.path.join(PROJECT_DIR, 'Data')
save_path_results = os.path.join(PROJECT_DIR, 'Results')
graphs_parent_dir = os.path.join(PROJECT_DIR, 'graphs')
graph_dir_name = 'KOLF_V4'
graph_path = os.path.join(graphs_parent_dir, graph_dir_name)
os.makedirs(data_path, exist_ok=True)
os.makedirs(save_path_results, exist_ok=True)
os.makedirs(graph_path, exist_ok=True)

FULL_INPUT_NAME = 'KOLF_V4_full_QC.h5ad'
HVG_INPUT_NAME = 'KOLF_V4_control_HVG5000.h5ad'
GRAPH_INPUT_NAME = 'KOLF_V4_graph_input.h5ad'
SEACELL_OUTPUT_NAME = 'SEACell_soft_ad_KOLF_V4.h5ad'
CELLS_OUTPUT_NAME = 'cells_KOLF_V4.pkl'
TARGET_COVERAGE_NAME = 'V4_VCC_target_coverage.csv'
OUTPUT_ORDER_NAME = 'V4_output_gene_order.csv'
GRAPH_GENES_NAME = 'V4_graph_genes.csv'
GRAPH_NODE_MAPPING_NAME = 'V4_graph_node_mapping.csv'
TRAIN_DEGS_NAME = 'V4_train_DEGs.csv'
OOD_SPLIT_NAME = 'V4_ood_split.json'
GRAPH_MANIFEST_NAME = 'V4_graph_manifest.json'

# OOD 选择：
# - 'kolf_top'：按 KOLF 扰动细胞数选当前实验 OOD，保持与已有 VDR/ZFP41 对比；
# - 'vcc_intersection'：从 VCC 有直接 KOLF 监督的靶点中选，更贴近最终比赛；
# - 设置 TESTING_GENES 时优先使用指定靶点。
OOD_SELECTION_MODE = 'kolf_top'
TESTING_GENES = []
N_TESTING_GENES = 2
DEG_TOP_N = 100
DEG_PADJ_CUTOFF = 0.05
DEG_MIN_CELLS = 10
CORRELATION_PERCENTILE = 99.0
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
sc.settings.seed = RANDOM_SEED

## Import the data

In [ ]:
# 读取 V4 Data 的固定输出。
full_adata = sc.read_h5ad(os.path.join(data_path, FULL_INPUT_NAME))
hvg_adata = sc.read_h5ad(os.path.join(data_path, HVG_INPUT_NAME))
target_coverage_df = pd.read_csv(os.path.join(data_path, TARGET_COVERAGE_NAME))
output_order_df = pd.read_csv(os.path.join(data_path, OUTPUT_ORDER_NAME))

if not full_adata.obs_names.equals(hvg_adata.obs_names):
    raise ValueError('V4 full_QC 与 control_HVG 的细胞顺序不一致。')
if not {'condition', 'cell_type', 'cov_cond'}.issubset(full_adata.obs.columns):
    raise KeyError('V4 Data 输出缺少 condition/cell_type/cov_cond。')
if full_adata.obs.cell_type.nunique() != 1:
    raise ValueError('V4 Graph 按单一细胞类型设计。')

hvg_genes = hvg_adata.var_names.astype(str).tolist()
full_gene_set = set(full_adata.var_names.astype(str))
vcc_output_gene_set = set(output_order_df.gene_name.astype(str))
print('Full-QC:', full_adata.shape)
print('Control HVG:', hvg_adata.shape)
print('VCC output genes:', len(vcc_output_gene_set))

In [ ]:
# =============================================================================
# OOD split：保存后由 V4 Training 直接读取，保证 Graph/Training 完全一致
# =============================================================================
condition_counts = full_adata.obs.loc[
    full_adata.obs.condition != 'control', 'condition'
].value_counts()
all_kolf_targets = condition_counts.index.astype(str).tolist()
def csv_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return series.astype(str).str.lower().isin(['true', '1'])

configured_genes = [str(g) for g in TESTING_GENES]
missing_configured = [g for g in configured_genes if g not in condition_counts.index]
if missing_configured:
    raise ValueError(f'TESTING_GENES 在当前数据中不存在：{missing_configured}')

if configured_genes:
    testing_genes = configured_genes
    selection_rule = 'configured TESTING_GENES'
elif OOD_SELECTION_MODE == 'vcc_intersection':
    # 只从 VCC 靶点且当前 KOLF 有直接扰动监督的集合中选择。
    vcc_direct_targets = target_coverage_df.loc[
        csv_bool(target_coverage_df.node_feature_available)
        & csv_bool(target_coverage_df.in_kolf_condition),
        'target_gene'
    ].astype(str).tolist()
    candidates = [g for g in vcc_direct_targets if g in condition_counts.index]
    candidate_counts = condition_counts.loc[candidates]
    testing_genes = candidate_counts.head(N_TESTING_GENES).index.astype(str).tolist()
    selection_rule = 'top condition counts within VCC targets with direct KOLF supervision'
elif OOD_SELECTION_MODE == 'kolf_top':
    testing_genes = condition_counts.head(N_TESTING_GENES).index.astype(str).tolist()
    selection_rule = 'top condition counts among all KOLF targets'
else:
    raise ValueError("OOD_SELECTION_MODE 必须是 'kolf_top' 或 'vcc_intersection'。")

if not testing_genes:
    raise ValueError('没有可用于 OOD 的扰动靶点。')
training_targets = sorted(set(all_kolf_targets) - set(testing_genes))
if not training_targets:
    raise ValueError('排除 OOD 后没有训练靶点。')

split_info = {
    'version': 'V4',
    'testing_genes': testing_genes,
    'training_targets': training_targets,
    'n_testing_genes': len(testing_genes),
    'selection_rule': selection_rule,
    'ood_selection_mode': OOD_SELECTION_MODE,
    'random_seed': RANDOM_SEED,
}
with open(os.path.join(data_path, OOD_SPLIT_NAME), 'w', encoding='utf-8') as handle:
    json.dump(split_info, handle, ensure_ascii=False, indent=2)
print('OOD testing genes:', testing_genes)
print('Training targets:', len(training_targets))
print('OOD split saved to:', os.path.join(data_path, OOD_SPLIT_NAME))

In [ ]:
# =============================================================================
# 训练靶点 DEG：只使用 training targets，绝不使用 OOD perturbed 细胞
# =============================================================================
deg_targets = [g for g in training_targets if int(condition_counts.get(g, 0)) >= DEG_MIN_CELLS]
deg_skipped_low_cell = [g for g in training_targets if g not in deg_targets]
train_deg_records = []

if deg_targets:
    sc.tl.rank_genes_groups(
        full_adata,
        groupby='condition',
        groups=deg_targets,
        reference='control',
        method='t-test',
        rankby_abs=True,
        corr_method='benjamini-hochberg',
        n_genes=min(DEG_TOP_N, full_adata.n_vars),
    )
    for target in deg_targets:
        dedf = sc.get.rank_genes_groups_df(full_adata, group=target).copy()
        dedf['names'] = dedf['names'].astype(str)
        dedf['pvals_adj'] = pd.to_numeric(dedf['pvals_adj'], errors='coerce')
        dedf = dedf[
            dedf['names'].isin(vcc_output_gene_set)
            & dedf['pvals_adj'].notna()
            & (dedf['pvals_adj'] <= DEG_PADJ_CUTOFF)
        ].head(DEG_TOP_N)
        for rank, row in enumerate(dedf.itertuples(index=False), start=1):
            train_deg_records.append({
                'target_gene': target,
                'deg_rank': rank,
                'gene_name': str(row.names),
                'scores': float(getattr(row, 'scores', np.nan)),
                'logfoldchanges': float(getattr(row, 'logfoldchanges', np.nan)),
                'pvals_adj': float(row.pvals_adj),
            })
    del full_adata.uns['rank_genes_groups']
else:
    print('没有满足最少细胞数的训练靶点；本次不添加 DEG 节点。')

train_degs_df = pd.DataFrame(
    train_deg_records,
    columns=['target_gene', 'deg_rank', 'gene_name', 'scores', 'logfoldchanges', 'pvals_adj']
)
train_degs_df.to_csv(os.path.join(data_path, TRAIN_DEGS_NAME), index=False)
train_deg_genes = list(dict.fromkeys(train_degs_df.gene_name.astype(str).tolist()))
print('DEG targets used:', len(deg_targets))
print('DEG targets skipped for low cell count:', len(deg_skipped_low_cell))
print('Selected target-DEG pairs:', len(train_degs_df))
print('Unique train-DEG genes:', len(train_deg_genes))

In [ ]:
# =============================================================================
# 扩展图节点：HVG + VCC targets + 全部 KOLF 靶基因 + train DEGs
# =============================================================================
def csv_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return series.astype(str).str.lower().isin(['true', '1'])

vcc_node_genes = target_coverage_df.loc[
    csv_bool(target_coverage_df.node_feature_available), 'target_gene'
].astype(str).tolist()
vcc_node_genes = [g for g in vcc_node_genes if g in full_gene_set]
kolf_target_node_genes = [g for g in all_kolf_targets if g in full_gene_set]

def ordered_union(*gene_lists):
    return list(dict.fromkeys(
        gene for gene_list in gene_lists for gene in gene_list
    ))

graph_genes = ordered_union(
    hvg_genes,
    vcc_node_genes,
    kolf_target_node_genes,
    train_deg_genes,
)
missing_graph_genes = [g for g in graph_genes if g not in full_gene_set]
if missing_graph_genes:
    raise ValueError(f'图节点不在 full_QC 中：{missing_graph_genes[:20]}')

hvg_set = set(hvg_genes)
vcc_node_set = set(vcc_node_genes)
kolf_target_node_set = set(kolf_target_node_genes)
train_deg_set = set(train_deg_genes)
graph_genes_df = pd.DataFrame({
    'graph_index': np.arange(len(graph_genes), dtype=int),
    'gene_name': graph_genes,
    'source_control_hvg': [g in hvg_set for g in graph_genes],
    'source_vcc_target': [g in vcc_node_set for g in graph_genes],
    'source_kolf_target_node': [g in kolf_target_node_set for g in graph_genes],
    'source_train_deg': [g in train_deg_set for g in graph_genes],
    'in_vcc_output_panel': [g in vcc_output_gene_set for g in graph_genes],
})
for col in ['control_detection_rate', 'control_mean_log1p']:
    if col in full_adata.var.columns:
        lookup = full_adata.var[col].copy()
        lookup.index = full_adata.var_names.astype(str)
        graph_genes_df[col] = [float(lookup.get(g, 0.0)) for g in graph_genes]

graph_genes_df.to_csv(os.path.join(data_path, GRAPH_GENES_NAME), index=False)
adata = full_adata[:, graph_genes].copy()
adata.var['graph_index'] = np.arange(adata.n_vars, dtype=int)
adata.var['source_control_hvg'] = adata.var_names.isin(hvg_set)
adata.var['source_vcc_target'] = adata.var_names.isin(vcc_node_set)
adata.var['source_kolf_target_node'] = adata.var_names.isin(kolf_target_node_set)
adata.var['source_train_deg'] = adata.var_names.isin(train_deg_set)

print('Graph genes:', len(graph_genes))
print('  control HVG:', len(hvg_genes))
print('  VCC target nodes:', len(vcc_node_genes))
print('  KOLF target nodes (train + OOD, control-only):', len(kolf_target_node_genes))
print('  unique train-DEG nodes:', len(train_deg_genes))
print('Graph input shape:', adata.shape)

## Constructing cell type graphs over control cells 

In [ ]:
ctrl_data = adata[adata.obs.condition == 'control'].copy()
ctrl_data.obs.cell_type.value_counts()

In [ ]:
fig, axes = plt.subplots(nrows=2, figsize=(6, 6))
sc.pl.umap(
    adata=hvg_adata,
    color=['perturbation_status'],
    legend_fontsize=18,
    ax=axes[0],
    title='Condition',
    show=False,
    palette=sns.color_palette('hls', 2),
    frameon=False,
)
sc.pl.umap(
    adata=hvg_adata,
    color=['cell_type'],
    legend_fontsize=18,
    ax=axes[1],
    title='Cell Line',
    show=False,
    palette=sns.color_palette('hls', len(hvg_adata.obs['cell_type'].unique())),
    frameon=False,
)
for ax in axes:
    ax.tick_params(labelsize=16)
plt.tight_layout()
plt.savefig(os.path.join(save_path_results, 'KOLF_V4_graph_umap.pdf'), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
ctrl_data.obs.cell_type.value_counts()

In [ ]:
# SEACells 仅基于 control；PCA 也只在 control 上重新计算。
if len(ctrl_data) < 3:
    raise ValueError('control 细胞少于 3 个，无法稳定运行 SEACells。')
n_pcs = min(50, ctrl_data.n_obs - 1, ctrl_data.n_vars - 1)
sc.pp.pca(ctrl_data, n_comps=n_pcs, random_state=RANDOM_SEED)
# SEACells 初始化图默认读取 X_umap；扩展 graph_input 没有该坐标。
# 直接复用 V4 Data 在 control HVG 上已计算好的 UMAP，仅用于可视化。
if 'X_umap' not in ctrl_data.obsm:
    if 'X_umap' not in hvg_adata.obsm:
        raise KeyError('hvg_adata 缺少 X_umap；请先完整运行 V4_KOLF_Data.ipynb。')
    ctrl_data.obsm['X_umap'] = hvg_adata[ctrl_data.obs_names].obsm['X_umap'].copy()

n_SEACells = max(2, min(int(len(ctrl_data) / 50), len(ctrl_data) - 1))
print('n_SEACells:', n_SEACells)
build_kernel_on = 'X_pca'
n_waypoint_eigs = min(10, n_SEACells - 1)

## Run SEACells over the control data.

In [ ]:
model = SEACells.core.SEACells(ctrl_data, 
                  build_kernel_on=build_kernel_on, 
                  n_SEACells=n_SEACells, 
                  n_waypoint_eigs=n_waypoint_eigs,
                  convergence_epsilon = 1e-5)

In [ ]:
model.construct_kernel_matrix()
M = model.kernel_matrix


In [ ]:
# Initialize archetypes
model.initialize_archetypes()

In [ ]:
# Plot the initilization to ensure they are spread across phenotypic space
SEACells.plot.plot_initialization(ctrl_data, model)

In [ ]:
model.fit(min_iter=10, max_iter=100)

In [ ]:
# You can force the model to run additional iterations step-wise using the .step() function
print(f'Ran for {len(model.RSS_iters)} iterations')
for _ in range(5):
    model.step()
print(f'Ran for {len(model.RSS_iters)} iterations')

In [ ]:
# Check for convergence 
model.plot_convergence()

In [ ]:
ctrl_data.obs[['SEACell']].head()
ctrl_data.obs.cell_type

In [ ]:
ctrl_data.obs[['SEACell']].head()

In [ ]:
adata.obs['SEACell'] = 'None'
adata.obs.loc[ctrl_data.obs.index.tolist(), 'SEACell'] = ctrl_data.obs['SEACell'].values

In [ ]:
ctrl_data.layers['X'] = ctrl_data.X

## Summarize SEACells by cell type label

In [ ]:
SEACell_soft_ad = SEACells.core.summarize_by_soft_SEACell(
    ctrl_data, model.A_,
    celltype_label='cell_type',
    summarize_layer='counts',
    minimum_weight=0.05,
)
print(SEACell_soft_ad.X.toarray() if issparse(SEACell_soft_ad.X) else SEACell_soft_ad.X)

In [ ]:
SEACell_soft_ad.obs

In [ ]:
SEACell_soft_ad.obs
SEACell_soft_ad.obs.rename({"celltype": "cell_type"}, axis=1, inplace=True)

In [ ]:
SEACell_soft_ad.obs.cell_type.value_counts()

In [ ]:
SEACell_soft_ad.write(os.path.join(data_path, SEACELL_OUTPUT_NAME))

In [ ]:
# 将 SEACell 标签写回 V4 扩展图输入数据。
adata.write(os.path.join(data_path, GRAPH_INPUT_NAME))

In [ ]:
SEACell_soft_ad = sc.read_h5ad(os.path.join(data_path, SEACELL_OUTPUT_NAME))
SEACell_soft_ad

In [ ]:
SEACell_soft_ad.obs.cell_type.value_counts()

## Construct co-expression graphs using SEACells generated from the control data.

In [ ]:
import os
import networkx as nx
from torch_geometric.utils.convert import from_networkx
from sklearn.metrics import mean_squared_error
from torch_geometric.data import InMemoryDataset, Data, download_url, extract_zip, HeteroData, Batch
from torch_geometric.utils import *
import torch
from torch import nn

# Kang 的 gene_name 是其基因索引；KOLF 同样显式提供该列给原 utils 函数使用。
SEACell_soft_ad.var['gene_name'] = SEACell_soft_ad.var_names.astype(str)

cell_type_network = {}
path = graph_path
print(path)
os.makedirs(path, exist_ok=True)

# 仅清理 V4 图目录中的旧图，防止换数据后混入过期文件。
for old_file in os.listdir(path):
    if old_file.endswith('_coexpr_graph.pkl'):
        os.remove(os.path.join(path, old_file))

max_nodes = 0
genes_pos = []
graph_node_records = []
for cell_type in SEACell_soft_ad.obs.cell_type.unique():
    print(cell_type)
    cell = Correlation_matrix(
        SEACell_soft_ad, cell_type, 'cell_type',
        hv_genes_cells=None, union_HVGs=True
    )
    threshold = np.percentile(np.abs(cell[0].values), CORRELATION_PERCENTILE)
    print("threshold:", threshold)
    g = create_coexpression_graph(
        SEACell_soft_ad, cell, cell_type, threshold, 'gene_name'
    )
    if g.num_nodes >= max_nodes:
        max_nodes = g.num_nodes
    cell_type_network[cell_type] = g
    genes_pos.append(cell_type_network[cell_type].pos)
    cell_type_network[cell_type].pos = torch.tensor(cell_type_network[cell_type].pos)
    for node_index, graph_index in enumerate(cell_type_network[cell_type].pos.tolist()):
        graph_node_records.append({
            'cell_type': cell_type,
            'node_index': node_index,
            'graph_index': int(graph_index),
            'gene_name': str(adata.var_names[int(graph_index)]),
        })
    torch.save(
        cell_type_network[cell_type],
        os.path.join(path, cell_type + '_coexpr_graph.pkl')
    )
cell_type_network
graph_node_mapping_df = pd.DataFrame(graph_node_records)
graph_node_mapping_df.to_csv(os.path.join(data_path, GRAPH_NODE_MAPPING_NAME), index=False)
print('Actual graph-node mapping saved:', len(graph_node_mapping_df))

## Check the overlapping genes between the cell type graphs

In [ ]:
# 保留 Kang 的图节点重叠矩阵；KOLF 只有一个细胞类型，因此是 1 × 1。
overlap_df = pd.DataFrame(
    index=cell_type_network.keys(),
    columns=cell_type_network.keys()
)
for key1 in cell_type_network.keys():
    for key2 in cell_type_network.keys():
        overlap_df.loc[key1, key2] = len(np.intersect1d(
            cell_type_network[key1].pos,
            cell_type_network[key2].pos
        ))

# Kang 用重叠关系选择 testing cell type；单细胞类型时直接取唯一类别。
if len(overlap_df.columns) > 1:
    n_select = min(7, len(overlap_df))
    indices = np.argsort(overlap_df.values, axis=0)[-n_select, :]
    testing_cell = overlap_df.columns[mode(indices)]
else:
    indices = np.array([0])
    testing_cell = overlap_df.columns[0]

print(indices)
print(testing_cell)
overlap_df

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
sns.set()
sns.set_style({'axes.grid': False})
cmap = sns.light_palette("#1C2E54", as_cmap=True)

# 多类别沿用 Kang 的 clustermap；单类别用普通热图，避免单样本聚类报错。
if overlap_df.shape[0] > 1:
    g = sns.clustermap(
        overlap_df.astype(float), figsize=(6, 6), cmap=cmap,
        cbar_kws={'label': 'overlapped genes', 'location': 'left'}
    )
    g.fig.subplots_adjust(right=0.7)
    g.ax_cbar.set_position((-0.1, .52, .03, .4))
else:
    fig, ax = plt.subplots(figsize=(6, 6))
    sns.heatmap(
        overlap_df.astype(float), cmap=cmap, annot=True, fmt='.0f',
        cbar_kws={'label': 'overlapped genes'}, ax=ax
    )
    ax.set_title('KOLF graph node overlap')

plt.savefig(os.path.join(save_path_results, 'overlap_matrix_KOLF_V4.pdf'), bbox_inches='tight')
plt.show()
sns.set()

In [ ]:
adata = sc.read_h5ad(os.path.join(data_path, GRAPH_INPUT_NAME))
adata.obs['cov_cond'] = (
    adata.obs.cell_type.astype(str) + '_' + adata.obs.condition.astype(str)
)
adata.obs['cov_drug'] = adata.obs['cov_cond']
adata

In [ ]:
print(len(adata.obs.loc[adata.obs.condition == 'control']), len(adata.obs.loc[adata.obs.condition != 'control']))
print(len(adata.obs.loc[adata.obs.condition == 'control']) + len(adata.obs.loc[adata.obs.condition != 'control']))

In [ ]:
import os
cell_type_network = {}
path = graph_path
print(path)
os.makedirs(path, exist_ok=True)

max_nodes = 0
genes_pos = []
for cell_type in adata.obs.cell_type.unique():
    print(cell_type)
    cell_type_network[cell_type] = torch.load(
        os.path.join(path, cell_type + '_coexpr_graph.pkl')
    )
    if cell_type_network[cell_type].num_nodes >= max_nodes:
        max_nodes = cell_type_network[cell_type].num_nodes
print(max_nodes)
cell_type_network

## Match treatment and control cells using SEACell clusters

In [ ]:
# 保留 Kang：为每个扰动细胞匹配相同 SEACell 分层的对照细胞。
def to_dense(matrix):
    return matrix.toarray() if hasattr(matrix, 'toarray') else np.asarray(matrix)

stim_data = adata[adata.obs.condition != 'control'].copy()
stim_data.layers['ctrl_x'] = to_dense(stim_data.X)
ctrl_data = adata[adata.obs.condition == 'control'].copy()

for cell_type in adata.obs.cell_type.unique():
    genes = cell_type_network[cell_type].pos.tolist()
    ctrl_expr = to_dense(
        ctrl_data[ctrl_data.obs.cell_type == cell_type, genes].copy().X
    )
    cell_type_network[cell_type].x = torch.mean(
        torch.tensor(ctrl_expr), dim=0
    )
    std = torch.var(torch.tensor(ctrl_expr), dim=0)
    cell_type_network[cell_type].x = torch.cat([
        cell_type_network[cell_type].x.unsqueeze(1), std.unsqueeze(1)
    ], dim=1)

for cov_drug in tq.tqdm(stim_data.obs.cov_drug.unique()):
    # cov_drug 仅为兼容字段；split 得到的是细胞类型和靶基因。
    cell_type = cov_drug.rsplit('_', 1)[0]
    ad_ctrl = ctrl_data[ctrl_data.obs.cell_type == cell_type].copy()
    ad_stim = stim_data[stim_data.obs.cov_drug == cov_drug].copy()
    num_samples = len(ad_stim)
    sele_samples = balance_subsample(
        np.array(ad_ctrl.obs.index.values.tolist()),
        np.array(ad_ctrl.obs.SEACell.values.tolist()),
        num_samples
    )
    matched_ctrl_x = to_dense(ad_ctrl[sele_samples, :].copy().X)
    row_indices = stim_data.obs_names.get_indexer(ad_stim.obs_names)
    stim_data.layers['ctrl_x'][row_indices, :] = matched_ctrl_x

## Construct PyTorch Geometric (PyG) data objects for the cells to enable GNN training.

In [ ]:
from importlib import reload
import utils
reload(utils)
from utils import *
cells_train = create_cells(stim_data, cell_type_network, None)

In [ ]:
import pickle
with open(os.path.join(data_path, CELLS_OUTPUT_NAME), 'wb') as f:
    pickle.dump(cells_train, f)
print('Saved:', os.path.join(data_path, CELLS_OUTPUT_NAME))

In [ ]:
adata.write(os.path.join(data_path, GRAPH_INPUT_NAME))
adata

In [ ]:
# =============================================================================
# V4 graph manifest：记录节点来源、OOD 隔离和实际图规模
# =============================================================================
graph_summaries = {}
for cell_type, network in cell_type_network.items():
    pos = network.pos.tolist()
    if len(pos) != int(network.num_nodes):
        raise ValueError(f'{cell_type} 的 graph.pos 数量与 num_nodes 不一致。')
    if pos and (min(pos) < 0 or max(pos) >= adata.n_vars):
        raise ValueError(f'{cell_type} 的 graph.pos 超出 graph_input 基因范围。')
    graph_summaries[str(cell_type)] = {
        'num_nodes': int(network.num_nodes),
        'num_edges': int(network.edge_index.shape[1]),
        'file': os.path.join(graph_path, str(cell_type) + '_coexpr_graph.pkl'),
    }

graph_manifest = {
    'version': 'V4',
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'random_seed': RANDOM_SEED,
    'inputs': {
        'full_qc': os.path.join(data_path, FULL_INPUT_NAME),
        'control_hvg': os.path.join(data_path, HVG_INPUT_NAME),
        'target_coverage': os.path.join(data_path, TARGET_COVERAGE_NAME),
        'output_gene_order': os.path.join(data_path, OUTPUT_ORDER_NAME),
    },
    'ood': {
        'testing_genes': testing_genes,
        'n_training_targets': len(training_targets),
        'ood_excluded_from_deg_selection': True,
        'ood_target_nodes_use_control_expression_only': True,
    },
    'deg_selection': {
        'top_n_per_training_target': DEG_TOP_N,
        'p_adj_cutoff': DEG_PADJ_CUTOFF,
        'min_cells_per_target': DEG_MIN_CELLS,
        'targets_used': deg_targets,
        'targets_skipped_low_cell_count': deg_skipped_low_cell,
        'n_target_deg_pairs': int(len(train_degs_df)),
        'n_unique_deg_genes': len(train_deg_genes),
        'restricted_to_vcc_output_panel': True,
    },
    'graph_genes': {
        'n_total': len(graph_genes),
        'n_control_hvg': len(hvg_genes),
        'n_vcc_target_nodes': len(vcc_node_genes),
        'n_kolf_target_nodes_train_and_ood': len(kolf_target_node_genes),
        'n_unique_train_deg_nodes': len(train_deg_genes),
        'correlation_percentile': CORRELATION_PERCENTILE,
    },
    'seacells': {
        'n_seacells': int(n_SEACells),
        'pca_and_seacells_use_control_only': True,
    },
    'graphs': graph_summaries,
    'outputs': {
        'graph_input_h5ad': os.path.join(data_path, GRAPH_INPUT_NAME),
        'seacell_soft_h5ad': os.path.join(data_path, SEACELL_OUTPUT_NAME),
        'cells_pickle': os.path.join(data_path, CELLS_OUTPUT_NAME),
        'graph_genes_csv': os.path.join(data_path, GRAPH_GENES_NAME),
        'graph_node_mapping_csv': os.path.join(data_path, GRAPH_NODE_MAPPING_NAME),
        'train_degs_csv': os.path.join(data_path, TRAIN_DEGS_NAME),
        'ood_split_json': os.path.join(data_path, OOD_SPLIT_NAME),
        'graph_directory': graph_path,
    },
    'software': {
        'python': sys.version,
        'scanpy': sc.__version__,
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'torch': torch.__version__,
    },
}
graph_manifest_path = os.path.join(data_path, GRAPH_MANIFEST_NAME)
with open(graph_manifest_path, 'w', encoding='utf-8') as handle:
    json.dump(graph_manifest, handle, ensure_ascii=False, indent=2)
print('V4 graph manifest saved to:', graph_manifest_path)
print(json.dumps({
    'graph_genes': graph_manifest['graph_genes'],
    'graphs': graph_summaries,
}, ensure_ascii=False, indent=2))